# Caso 01 — BrightCart (Moda Online): «Lo que vuelve» (Análisis Forense de Devoluciones)
## Día 4 — De la Facturación Bruta al Margen Neto Real

**Situación crítica:** Nuria (Marketing) tiene lista la orden de compra de **2.000 unidades de reposición del *Vestido Alma*** para agosto tras ver sus `1.016` prendas vendidas en el Día 3.

### Misión del Día 4 junto a Diego (Head de Datos):
1. Cruzar mediante `LEFT JOIN` las `17.044` líneas vendidas en las Rebajas de Verano con `devoluciones_limpias.csv` para medir la **Tasa de Devolución Global** y el **Ingreso Neto Real**.
2. Auditar la sangría por **categoría** y demostrar cómo `Vestidos` y `Calzado` concentran más del `60 %` de los reembolsos.
3. Realizar la **autopsia forense del `Vestido Alma`** por **talla** y por **motivo de devolución** para aislar la causa raíz.
4. Recalcular el **ROAS Neto de Instagram Ads** y cuantificar el **coste oculto de la logística inversa (`6,50 €/prenda devuelta`)**.

In [1]:
# =============================================================================
# CARGA DE DATOS LIMPIOS Y CATÁLOGO CON CATEGORÍAS NORMALIZADAS
# =============================================================================
import pandas as pd
import numpy as np

lineas = pd.read_csv('data/clean/lineas_limpias.csv')
pedidos = pd.read_csv('data/clean/pedidos_limpios.csv')
devoluciones = pd.read_csv('data/clean/devoluciones_limpias.csv')
productos = pd.read_csv('data/productos.csv')

def unificar_categoria(c):
    c = str(c).strip().lower().replace('é', 'e')
    if 'vestid' in c: return 'Vestidos'
    if 'calzad' in c or 'zapat' in c: return 'Calzado'
    if 'abrig' in c: return 'Abrigos'
    if 'pantalon' in c: return 'Pantalones'
    if 'camiset' in c: return 'Camisetas'
    if 'fald' in c: return 'Faldas'
    if 'jers' in c: return 'Jerséis'
    if 'accesorio' in c or 'complement' in c: return 'Accesorios'
    return c.title()

productos['categoria'] = productos['categoria'].apply(unificar_categoria)
print('Datos limpios cargados y categorías de productos normalizadas.')

Datos limpios cargados y categorías de productos normalizadas.


### Análisis de Preparación
* Las tres tablas limpias (`lineas_limpias`, `pedidos_limpios`, `devoluciones_limpias`) y el catálogo con las 8 categorías unificadas están cargados en memoria.

---
## Reto 1 (Día 4): El Cruce Forense (`LEFT JOIN` de Ventas vs. Devoluciones)
Unimos las ventas de las Rebajas de Verano (`campana_id == 1`) con `devoluciones` usando `how='left'` para conservar el `100 %` de las líneas vendidas (`17.044`) y marcar cuáles acabaron de vuelta en el almacén.

In [2]:
# =============================================================================
# RETO 1 (DÍA 4): CONSTRUCCIÓN DEL DATASET MAESTRO CON DEVOLUCIONES
# =============================================================================
df_verano = (
    lineas.merge(pedidos, on='pedido_id', how='inner')
    .merge(productos, on='producto_id', how='inner')
)
df_verano = df_verano[df_verano['campana_id'] == 1].copy()

df_master = df_verano.merge(
    devoluciones[['linea_id', 'motivo', 'fecha_devolucion']],
    on='linea_id',
    how='left'
)

df_master['devuelto'] = df_master['motivo'].notna()

total_lineas = len(df_master)
lineas_devueltas = int(df_master['devuelto'].sum())
tasa_dev_global = (lineas_devueltas / total_lineas) * 100
unidades_vendidas = int(df_master['cantidad'].sum())
unidades_devueltas = int(df_master.loc[df_master['devuelto'], 'cantidad'].sum())
dinero_bruto = df_master['importe_linea'].sum()
dinero_reembolsado = df_master.loc[df_master['devuelto'], 'importe_linea'].sum()
dinero_neto_real = dinero_bruto - dinero_reembolsado
pct_dinero_devuelto = (dinero_reembolsado / dinero_bruto) * 100

print('--- DIAGNÓSTICO FORENSE GLOBAL DE DEVOLUCIONES (CAMPAÑA VERANO) ---')
print(f'Total Líneas Vendidas:        {total_lineas:,} ({unidades_vendidas:,} unidades)')
print(f'Total Líneas Devueltas:       {lineas_devueltas:,} ({unidades_devueltas:,} unidades)')
print(f'Tasa de Devolución Global:    {tasa_dev_global:.2f}%')
print(f'Ingresos Brutos en Caja:      {dinero_bruto:,.2f} €')
print(f'Dinero Devuelto (Reembolsos): {dinero_reembolsado:,.2f} € ({pct_dinero_devuelto:.2f}% del bruto)')
print(f'Ingreso Neto Real Retenido:   {dinero_neto_real:,.2f} €')

--- DIAGNÓSTICO FORENSE GLOBAL DE DEVOLUCIONES (CAMPAÑA VERANO) ---
Total Líneas Vendidas:        17,044 (20,008 unidades)
Total Líneas Devueltas:       4,718 (5,535 unidades)
Tasa de Devolución Global:    27.68%
Ingresos Brutos en Caja:      572,301.82 €
Dinero Devuelto (Reembolsos): 169,918.32 € (29.69% del bruto)
Ingreso Neto Real Retenido:   402,383.51 €


### Análisis del Reto 1 (El Golpe de Realidad Macro)
* **Casi `3` de cada `10` compras volvieron al almacén:** De las `17.044` líneas vendidas en la campaña, **`4.718` fueron devueltas (`27,68 %` de tasa de devolución; `5.535` prendas devueltas de `20.008`)**.
* **En dinero, la sangría es aún mayor (`29,69 %`):** Como se devuelven más las prendas de mayor precio unitario (`Vestidos` y `Calzado`), BrightCart tuvo que reembolsar **`169.918,32 €`** de los `572.301,82 €` cobrados inicialmente, quedándose con un **Ingreso Neto Real de solo `402.383,51 €`** (un `-21,46 %` por debajo de los `512.340,00 €` de referencia del año anterior).

---
## Reto 2 (Día 4): Tasa de Devolución y Balance Neto por Categoría
Auditamos las 8 categorías para comprobar si el `27,68 %` de devolución es uniforme en toda la tienda o si está focalizado en familias específicas.

In [3]:
# =============================================================================
# RETO 2 (DÍA 4): TASA DE DEVOLUCIÓN Y BALANCE NETO POR CATEGORÍA
# =============================================================================
cat_devoluciones = (
    df_master.groupby('categoria')
    .agg(
        ventas=('devuelto', 'count'),
        unidades=('cantidad', 'sum'),
        devueltos=('devuelto', 'sum'),
        ingresos_brutos=('importe_linea', 'sum'),
        reembolsos=('importe_linea', lambda x: x[df_master.loc[x.index, 'devuelto']].sum())
    )
)

cat_devoluciones['tasa_dev_pct'] = (cat_devoluciones['devueltos'] / cat_devoluciones['ventas']) * 100
cat_devoluciones['ingresos_netos'] = cat_devoluciones['ingresos_brutos'] - cat_devoluciones['reembolsos']
cat_devoluciones['pct_del_total_reembolsos'] = (cat_devoluciones['reembolsos'] / cat_devoluciones['reembolsos'].sum()) * 100
cat_devoluciones = cat_devoluciones.sort_values(by='tasa_dev_pct', ascending=False)

print('Ranking de Devoluciones por Categoría (Rebajas de Verano):')
display(cat_devoluciones.style.format({
    'ventas': '{:,.0f}',
    'unidades': '{:,.0f}',
    'devueltos': '{:,.0f}',
    'tasa_dev_pct': '{:.2f}%',
    'ingresos_brutos': '{:,.2f} €',
    'reembolsos': '{:,.2f} €',
    'ingresos_netos': '{:,.2f} €',
    'pct_del_total_reembolsos': '{:.2f}%'
}))

Ranking de Devoluciones por Categoría (Rebajas de Verano):


,ventas,unidades,devueltos,ingresos_brutos,reembolsos,tasa_dev_pct,ingresos_netos,pct_del_total_reembolsos
categoria,,,,,,,,
Vestidos,"4,154","4,870","1,482","154,292.26 €","54,542.01 €",35.68%,"99,750.25 €",32.10%
Calzado,"2,667","3,134",909,"141,720.87 €","48,619.03 €",34.08%,"93,101.84 €",28.61%
Faldas,"1,463","1,701",474,"41,849.37 €","13,493.44 €",32.40%,"28,355.93 €",7.94%
Pantalones,"1,674","1,965",511,"54,457.69 €","16,455.00 €",30.53%,"38,002.69 €",9.68%
Abrigos,833,"1,000",220,"49,503.03 €","12,705.20 €",26.41%,"36,797.82 €",7.48%
Jerséis,"1,374","1,603",304,"44,342.38 €","10,041.88 €",22.13%,"34,300.50 €",5.91%
Camisetas,"2,831","3,330",559,"44,698.04 €","8,821.85 €",19.75%,"35,876.19 €",5.19%
Accesorios,"2,048","2,405",259,"41,438.19 €","5,239.90 €",12.65%,"36,198.28 €",3.08%


### Análisis del Reto 2 (Dónde Sangra el Negocio)
* **Las dos categorías líderes en ventas son también las líderes en devoluciones:**
  * **`Vestidos`** sufre una **tasa de devolución del `35,68 %`** (`1.482` líneas devueltas de `4.154`), perdiendo **`-54.542,01 €` en reembolsos** (`32,10 %` de todo el dinero devuelto en la campaña) y cayendo de `154.292,26 €` brutos a **`99.750,25 €` netos**.
  * **`Calzado`** le sigue con un **`34,08 %` de devolución** (`909` de `2.667`), perdiendo **`-48.619,03 €`** (`28,61 %` de los reembolsos) y cayendo de `141.720,87 €` brutos a **`93.101,84 €` netos**.
  * Entre **`Vestidos` y `Calzado` concentran `103.161,04 €` en reembolsos (`60,71 %` de toda la fuga de caja de la campaña)**.
* En el extremo sano, **`Accesorios` (`12,65 %` de devolución)** y **`Camisetas` (`19,75 %`)** retienen más del `80 %–87 %` de todo lo que facturan.

---
## Reto 3 (Día 4): La Trampa del «Vestido Alma» — Radiografía por Talla y Motivo
Hacemos un zoom clínico sobre el **`Vestido Alma` (`producto_id = 1042`)** y sobre el **Top 10 de productos** para verificar qué ocurre con su tasa de devolución, en qué **tallas** se rompe y qué **motivos** declaran las clientas.

In [4]:
# =============================================================================
# RETO 3 (DÍA 4): RADIOGRAFÍA FORENSE DEL 'VESTIDO ALMA' (TALLAS Y MOTIVOS)
# =============================================================================
alma = df_master[df_master['nombre'] == 'Vestido Alma'].copy()

alma_ventas = len(alma)
alma_unidades = int(alma['cantidad'].sum())
alma_devueltos = int(alma['devuelto'].sum())
alma_unidades_dev = int(alma.loc[alma['devuelto'], 'cantidad'].sum())
alma_tasa_dev = (alma_devueltos / alma_ventas) * 100
alma_bruto = alma['importe_linea'].sum()
alma_reembolsos = alma.loc[alma['devuelto'], 'importe_linea'].sum()
alma_neto = alma_bruto - alma_reembolsos

print('=== 1. REPORTE DE AUDITORÍA: VESTIDO ALMA ===')
print(f'Líneas de Venta:              {alma_ventas:,} ({alma_unidades:,} prendas vendidas)')
print(f'Líneas Devueltas:             {alma_devueltos:,} ({alma_unidades_dev:,} prendas devueltas)')
print(f'Tasa de Devolución Alma:      {alma_tasa_dev:.2f}%  (vs 35.68% Vestidos / 27.68% Global)')
print(f'Facturación Bruta:            {alma_bruto:,.2f} €')
print(f'Dinero Reembolsado:           {alma_reembolsos:,.2f} €')
print(f'Facturación Neta Real:        {alma_neto:,.2f} €\n')

# Desglose por Talla del Vestido Alma
tallas_alma = (
    alma.groupby('talla')
    .agg(
        lineas_vendidas=('devuelto', 'count'),
        unidades_vendidas=('cantidad', 'sum'),
        lineas_devueltas=('devuelto', 'sum'),
        ingresos_brutos=('importe_linea', 'sum'),
        reembolsos=('importe_linea', lambda x: x[alma.loc[x.index, 'devuelto']].sum())
    )
    .reindex(['XS', 'S', 'M', 'L', 'XL'])
)
tallas_alma['tasa_dev_pct'] = (tallas_alma['lineas_devueltas'] / tallas_alma['lineas_vendidas']) * 100
tallas_alma['pct_de_devoluciones_alma'] = (tallas_alma['lineas_devueltas'] / alma_devueltos) * 100

print('=== 2. DESGLOSE POR TALLA DEL VESTIDO ALMA ===')
display(tallas_alma.style.format({
    'lineas_vendidas': '{:,}',
    'unidades_vendidas': '{:,}',
    'lineas_devueltas': '{:,}',
    'tasa_dev_pct': '{:.2f}%',
    'pct_de_devoluciones_alma': '{:.2f}%',
    'ingresos_brutos': '{:,.2f} €',
    'reembolsos': '{:,.2f} €'
}))

# Análisis de Motivos de Devolución
motivos_alma = alma['motivo'].value_counts().reset_index()
motivos_alma.columns = ['motivo', 'cantidad']
motivos_alma['pct'] = (motivos_alma['cantidad'] / motivos_alma['cantidad'].sum()) * 100

print('\n=== 3. MOTIVOS DE DEVOLUCIÓN DECLARADOS PARA EL VESTIDO ALMA ===')
display(motivos_alma.style.format({'cantidad': '{:,}', 'pct': '{:.2f}%'}))

=== 1. REPORTE DE AUDITORÍA: VESTIDO ALMA ===
Líneas de Venta:              858 (1,016 prendas vendidas)
Líneas Devueltas:             366 (447 prendas devueltas)
Tasa de Devolución Alma:      42.66%  (vs 35.68% Vestidos / 27.68% Global)
Facturación Bruta:            30,473.90 €
Dinero Reembolsado:           13,407.32 €
Facturación Neta Real:        17,066.59 €

=== 2. DESGLOSE POR TALLA DEL VESTIDO ALMA ===


,lineas_vendidas,unidades_vendidas,lineas_devueltas,ingresos_brutos,reembolsos,tasa_dev_pct,pct_de_devoluciones_alma
talla,,,,,,,
XS,35,41,4,"1,229.75 €",149.97 €,11.43%,1.09%
S,134,153,18,"4,589.08 €",689.86 €,13.43%,4.92%
M,310,369,167,"11,067.79 €","6,148.77 €",53.87%,45.63%
L,249,294,149,"8,818.24 €","5,368.93 €",59.84%,40.71%
XL,130,159,28,"4,769.05 €","1,049.79 €",21.54%,7.65%



=== 3. MOTIVOS DE DEVOLUCIÓN DECLARADOS PARA EL VESTIDO ALMA ===


,motivo,cantidad,pct
0,talla incorrecta,220,60.11%
1,no me queda bien,72,19.67%
2,Talla,27,7.38%
3,me queda pequeño,18,4.92%
4,no me gusta,14,3.83%
5,defecto,8,2.19%
6,cambió de opinión,4,1.09%
7,color distinto,3,0.82%


### Análisis del Reto 3 (El Espejismo del `Vestido Alma` Desmontado con Datos)
* **Anomalía extrema de devolución (`42,66 %`):** El `Vestido Alma` sufre una tasa de retorno del **`42,66 %`** (`366` líneas devueltas de `858`, equivalentes a **`433` prendas devueltas de las `1.016` vendidas**), superando en **`+14,98 pp`** la media de la tienda (`27,68 %`) y en **`+6,98 pp`** la media de su propia categoría `Vestidos` (`35,68 %`).
* **Pérdida en caja:** De los `30.473,90 €` facturados en bruto, **`13.407,32 €` tuvieron que reembolsarse**, reduciendo el ingreso neto retenido a **`17.066,59 €`**. Solo este vestido explica el **`7,89 %` de todos los reembolsos de la compañía** entre 400 referencias del catálogo.
* **Causa raíz exacta (Patronaje defectuoso en tallas `M` y `L`):**
  1. **Por motivo:** **`337` de las `366` devoluciones (`92,08 %`)** se deben exclusivamente a problemas de **talla y ajuste** (`talla incorrecta` `220` / `60,11 %`, `no me queda bien` `72` / `19,67 %`, `Talla` `27` / `7,38 %`, `me queda pequeño` `18` / `4,92 %`). No es un problema de gusto ni de calidad de tela (`defecto` es apenas el `2,19 %`).
  2. **Por talla:** Mientras las tallas `XS` (`11,43 %`) y `S` (`13,43 %`) tienen devoluciones muy bajas, el patrón se rompe por completo en la **talla `M` (`53,87 %` de devolución, `167` de `310`)** y en la **talla `L` (`59,84 %` de devolución, `149` de `249`)**. **Las tallas `M` y `L` concentran el `86,34 %` (`316 / 366`) de todas las devoluciones del Vestido Alma**: el proveedor talló `M` y `L` con medidas más pequeñas de lo estándar.

---
## Reto 4 (Día 4): El ROAS Real de Marketing y Desempeño Neto por Canal
Auditamos cómo cambian los números de los 4 canales de adquisición cuando pasamos de **Facturación Bruta** a **Facturación Neta Real (post-devoluciones)**.

In [5]:
# =============================================================================
# RETO 4 (DÍA 4): RENDIMIENTO NETO POR CANAL Y ROAS REAL DE INSTAGRAM
# =============================================================================
canal_balance = (
    df_master.groupby('canal')
    .agg(
        pedidos=('pedido_id', 'nunique'),
        ventas=('devuelto', 'count'),
        devueltos=('devuelto', 'sum'),
        ingresos_brutos=('importe_linea', 'sum'),
        reembolsos=('importe_linea', lambda x: x[df_master.loc[x.index, 'devuelto']].sum())
    )
)

canal_balance['tasa_dev_pct'] = (canal_balance['devueltos'] / canal_balance['ventas']) * 100
canal_balance['ingresos_netos'] = canal_balance['ingresos_brutos'] - canal_balance['reembolsos']
canal_balance['ticket_neto'] = canal_balance['ingresos_netos'] / canal_balance['pedidos']
canal_balance = canal_balance.sort_values(by='ingresos_netos', ascending=False)

print('=== BALANCE FINANCIERO NETO POR CANAL DE MARKETING ===')
display(canal_balance.style.format({
    'pedidos': '{:,.0f}',
    'ventas': '{:,.0f}',
    'devueltos': '{:,.0f}',
    'tasa_dev_pct': '{:.2f}%',
    'ingresos_brutos': '{:,.2f} €',
    'reembolsos': '{:,.2f} €',
    'ingresos_netos': '{:,.2f} €',
    'ticket_neto': '{:,.2f} €'
}))

presupuesto_ig = 48000.0
ig_bruto = canal_balance.loc['instagram', 'ingresos_brutos']
ig_neto = canal_balance.loc['instagram', 'ingresos_netos']
roas_bruto = ig_bruto / presupuesto_ig
roas_neto = ig_neto / presupuesto_ig

print('=== AUDITORÍA DE EFICIENCIA PUBLICITARIA (INSTAGRAM ADS) ===')
print(f'Presupuesto invertido:   {presupuesto_ig:,.2f} €')
print(f'Ingreso Bruto reportado: {ig_bruto:,.2f} €  --> ROAS Bruto: {roas_bruto:.2f}x')
print(f'Ingreso Neto real:       {ig_neto:,.2f} €  --> ROAS Neto:  {roas_neto:.2f}x')
print(f'Impacto de Devoluciones: -{roas_bruto - roas_neto:.2f}x de retorno perdido (-{ig_bruto - ig_neto:,.2f} €)')

=== BALANCE FINANCIERO NETO POR CANAL DE MARKETING ===


,pedidos,ventas,devueltos,ingresos_brutos,reembolsos,tasa_dev_pct,ingresos_netos,ticket_neto
canal,,,,,,,,
instagram,"2,400","6,037","1,599","201,623.15 €","57,235.51 €",26.49%,"144,387.64 €",60.16 €
google,"1,714","4,277","1,250","142,900.62 €","45,084.13 €",29.23%,"97,816.49 €",57.07 €
email,"1,361","3,380",932,"113,265.66 €","32,867.23 €",27.57%,"80,398.43 €",59.07 €
directo,"1,344","3,350",937,"114,512.39 €","34,731.44 €",27.97%,"79,780.95 €",59.36 €


=== AUDITORÍA DE EFICIENCIA PUBLICITARIA (INSTAGRAM ADS) ===
Presupuesto invertido:   48,000.00 €
Ingreso Bruto reportado: 201,623.15 €  --> ROAS Bruto: 4.20x
Ingreso Neto real:       144,387.64 €  --> ROAS Neto:  3.01x
Impacto de Devoluciones: -1.19x de retorno perdido (-57,235.51 €)


### Análisis del Reto 4 (ROAS Bruto vs. ROAS Neto en Canales)
* **La ilusión óptica del ROAS Bruto en Instagram:** Los `201.623,15 €` brutos atribuibles a `instagram` sufren **`-57.235,51 €` en devoluciones (`26,49 %` de las líneas)**, reduciendo la facturación neta real a **`144.387,64 €`**. Sobre los `48.000,00 €` de presupuesto publicitario, el **ROAS cae de `4,20x` a `3,01x` (`-1,19x` de retorno evaporado)**.
* **Comportamiento transversal por canal:** La tasa de devolución es elevada en los cuatro canales (`google` `29,23 %`, `directo` `27,97 %`, `email` `27,57 %`, `instagram` `26,49 %`), lo que demuestra que el problema de devoluciones **no es culpa de un canal concreto, sino del producto (`Vestidos` y `Calzado`) que todos los canales están empujando**. Además, tras descontar devoluciones, **`email` (`80.398,43 €` netos) supera a `directo` (`79.780,95 €` netos)** y se consolida como el canal orgánico más rentable.

---
## Reto 5 (Día 4): El Coste Oculto de la Logística Inversa (`6,50 €` por devolución)
Una devolución no deja el marcador a cero: implica transporte de recogida, recepción en almacén, inspección, planchado, reetiquetado y comisiones de pasarela no devueltas. Operaciones cuantifica este **coste de logística inversa en `6,50 €` por cada línea/prenda devuelta**.

In [6]:
# =============================================================================
# RETO 5 (DÍA 4): COSTE OPERATIVO TOTAL DE LA LOGÍSTICA INVERSA
# =============================================================================
COSTE_UNITARIO_DEVOLUCION = 6.50  # Euros por devolución procesada (recogida + almacén)

total_prendas_devueltas = int(df_master['devuelto'].sum())
coste_logistico_global = total_prendas_devueltas * COSTE_UNITARIO_DEVOLUCION
reembolsos_globales = df_master.loc[df_master['devuelto'], 'importe_linea'].sum()
impacto_economico_total = reembolsos_globales + coste_logistico_global
facturacion_neta_post_logistica = (df_master['importe_linea'].sum() - reembolsos_globales) - coste_logistico_global

alma_prendas_devueltas = int(alma['devuelto'].sum())
coste_logistico_alma = alma_prendas_devueltas * COSTE_UNITARIO_DEVOLUCION
reembolsos_alma = alma.loc[alma['devuelto'], 'importe_linea'].sum()
impacto_total_alma = reembolsos_alma + coste_logistico_alma

print('=== IMPACTO ECONÓMICO TOTAL DE DEVOLUCIONES (CAMPAÑA VERANO) ===')
print(f'Total Devoluciones Procesadas:     {total_prendas_devueltas:,}')
print(f'Dinero Reembolsado a Clientes:     {reembolsos_globales:,.2f} €')
print(f'Coste Operativo Logística Inversa: {coste_logistico_global:,.2f} €  ({COSTE_UNITARIO_DEVOLUCION:.2f} €/devolución)')
print(f'IMPACTO ECONÓMICO TOTAL NEGATIVO:  {impacto_economico_total:,.2f} €')
print(f'Caja Neta tras Logística Inversa:  {facturacion_neta_post_logistica:,.2f} €\n')

print('=== SANGRÍA ESPECÍFICA DEL VESTIDO ALMA ===')
print(f'Devoluciones de Vestido Alma:      {alma_prendas_devueltas:,}')
print(f'Reembolsos directos de Alma:       {reembolsos_alma:,.2f} €')
print(f'Coste de Logística Inversa Alma:   {coste_logistico_alma:,.2f} €')
print(f'Pérdida Total por Devoluciones:    {impacto_total_alma:,.2f} €')
print(f'Peso de Alma sobre reembolsos:     {(reembolsos_alma / reembolsos_globales)*100:.2f}%')

=== IMPACTO ECONÓMICO TOTAL DE DEVOLUCIONES (CAMPAÑA VERANO) ===
Total Devoluciones Procesadas:     4,718
Dinero Reembolsado a Clientes:     169,918.32 €
Coste Operativo Logística Inversa: 30,667.00 €  (6.50 €/devolución)
IMPACTO ECONÓMICO TOTAL NEGATIVO:  200,585.32 €
Caja Neta tras Logística Inversa:  371,716.51 €

=== SANGRÍA ESPECÍFICA DEL VESTIDO ALMA ===
Devoluciones de Vestido Alma:      366
Reembolsos directos de Alma:       13,407.32 €
Coste de Logística Inversa Alma:   2,379.00 €
Pérdida Total por Devoluciones:    15,786.32 €
Peso de Alma sobre reembolsos:     7.89%


### Análisis del Reto 5 (El Agujero Operativo de la Logística Inversa)
* **Más de `200.000 €` de impacto negativo en la campaña:** A los **`169.918,32 €`** de ingresos reembolsados hay que sumar **`30.667,00 €`** quemados directamente en gastos de recogida y manipulación de las `4.718` devoluciones (`6,50 €` c/u), elevando el **impacto económico negativo total a `200.585,32 €`** y dejando la caja neta post-logística en **`371.716,51 €`**.
* **En el `Vestido Alma`:** Sus `366` devoluciones generaron **`13.407,32 €` en reembolsos** más **`2.379,00 €` en logística inversa** (**`15.786,32 €` de impacto negativo**), comiéndose más de la mitad de su facturación bruta antes siquiera de descontar el coste de fabricación (COGS) y el CAC publicitario.

---
## Conclusión Ejecutiva del Día 4
1. **El freno al pedido de 2.000 unidades estaba plenamente justificado:** Ordenar `2.000` unidades adicionales del `Vestido Alma` sin corregir el patronaje de las tallas `M` (`53,87 %` dev.) y `L` (`59,84 %` dev.) habría provocado **`~853` devoluciones futuras** y más de **`31.100 €` en reembolsos y logística inversa**.
2. **La diferencia entre facturar y ganar dinero:** La campaña pasó de `748.731,02 €` (catálogo) $ightarrow$ `572.301,82 €` (cobrado en caja) $ightarrow$ **`402.383,51 €` (ingreso neto post-reembolsos)** $ightarrow$ **`371.716,51 €` (caja neta tras logística inversa)**.
3. **Siguiente paso (Día 5):** Construir los **4 gráficos ejecutivos** en `reports/`, modelar los **Unit Economics** completos del `Vestido Alma` y presentar el **One-Pager Ejecutivo** ante el Comité.